# A guide Portfolio Optimization Environment

This notebook aims to provide an example of using PortfolioOptimizationEnv (or POE) to train a reinforcement learning model that learns to solve the portfolio optimization problem.

In this document, we will reproduce a famous architecture called EIIE (ensemble of identical independent evaluators), introduced in the following paper:

- Zhengyao Jiang, Dixing Xu, & Jinjun Liang. (2017). A Deep Reinforcement Learning Framework for the Financial Portfolio Management Problem. https://doi.org/10.48550/arXiv.1706.10059.

It's advisable to read it to understand the algorithm implemented in this notebook.

### Note
If you're using this environment, consider citing the following paper (in adittion to FinRL references):

- Caio Costa, & Anna Costa (2023). POE: A General Portfolio Optimization Environment for FinRL. In *Anais do II Brazilian Workshop on Artificial Intelligence in Finance* (pp. 132–143). SBC. https://doi.org/10.5753/bwaif.2023.231144.

```
@inproceedings{bwaif,
 author = {Caio Costa and Anna Costa},
 title = {POE: A General Portfolio Optimization Environment for FinRL},
 booktitle = {Anais do II Brazilian Workshop on Artificial Intelligence in Finance},
 location = {João Pessoa/PB},
 year = {2023},
 keywords = {},
 issn = {0000-0000},
 pages = {132--143},
 publisher = {SBC},
 address = {Porto Alegre, RS, Brasil},
 doi = {10.5753/bwaif.2023.231144},
 url = {https://sol.sbc.org.br/index.php/bwaif/article/view/24959}
}

```

## Installation and imports

To run this notebook in google colab, uncomment the cells below.

In [62]:
# # install finrl library
# !sudo apt install swig
# !pip install git+https://github.com/AI4Finance-Foundation/FinRL.git

In [63]:
## We also need to install quantstats, because the environment uses it to plot graphs
# !pip install quantstats

In [64]:
## Hide matplotlib warnings
# import warnings
# warnings.filterwarnings('ignore')

import logging
logging.getLogger('matplotlib.font_manager').disabled = True

#### Import the necessary code libraries

In [65]:
import torch

import numpy as np

from sklearn.preprocessing import MaxAbsScaler

from finrl.meta.preprocessor.yahoodownloader import YahooDownloader
from finrl.meta.preprocessor.preprocessors import GroupByScaler
from finrl.meta.env_portfolio_optimization.env_portfolio_optimization import PortfolioOptimizationEnv
from finrl.agents.portfolio_optimization.models import DRLAgent
from finrl.agents.portfolio_optimization.architectures import EIIE

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'

## Fetch data

In his paper, *Jiang et al* creates a portfolio composed by the top-11 cryptocurrencies based on 30-days volume. Since it's not specified when this classification was done, it's difficult to reproduce, so we will use a similar approach in the Brazillian stock market:

- We select top-10 stocks from Brazillian stock market;
- For simplicity, we disconsider stocks that have missing data for the days in period 2011-01-01 to 2019-12-31 (9 years);

In [66]:

TOP_BRL =['000001.SS', '399001.SZ', '603000.SS','000035.SZ','002261.SZ','000938.SZ','600547.SS','600756.SS','601899.SS','601988.SS']

In [72]:
print(len(TOP_BRL))

portfolio_raw_df = YahooDownloader(start_date = '2016-01-01',
                                end_date = '2024-10-25',
                                ticker_list = TOP_BRL).fetch_data()
portfolio_raw_df

10


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

Shape of DataFrame:  (21368, 8)


,date,open,high,low,close,volume,tic,day
0,2016-01-04,3536.589111,3538.688965,3295.740967,3296.258057,184400,000001.SS,0
1,2016-01-04,7.675000,7.675000,7.675000,7.492187,0,000035.SZ,0
2,2016-01-04,35.896500,35.896500,32.266762,31.373688,22430301,000938.SZ,0
3,2016-01-04,19.299999,19.365000,17.504999,17.414419,16787116,002261.SZ,0
4,2016-01-04,12650.719727,12659.410156,11625.410156,11625.909180,1058400,399001.SZ,0
...,...,...,...,...,...,...,...,...
21363,2024-10-24,28.230000,28.240000,27.520000,27.780001,37308114,600547.SS,3
21364,2024-10-24,14.860000,16.180000,14.710000,15.410000,38161759,600756.SS,3
21365,2024-10-24,17.600000,17.600000,17.320000,17.490000,153140704,601899.SS,3
21366,2024-10-24,4.940000,4.990000,4.920000,4.950000,134299060,601988.SS,3


In [73]:
portfolio_raw_df.groupby("tic").count()

,date,open,high,low,close,volume,day
tic,,,,,,,
000001.SS,2137,2137,2137,2137,2137,2137,2137
000035.SZ,2137,2137,2137,2137,2137,2137,2137
000938.SZ,2137,2137,2137,2137,2137,2137,2137
002261.SZ,2137,2137,2137,2137,2137,2137,2137
399001.SZ,2135,2135,2135,2135,2135,2135,2135
600547.SS,2137,2137,2137,2137,2137,2137,2137
600756.SS,2137,2137,2137,2137,2137,2137,2137
601899.SS,2137,2137,2137,2137,2137,2137,2137
601988.SS,2137,2137,2137,2137,2137,2137,2137


In [74]:
from finrl.meta.preprocessor.preprocessors import FeatureEngineer
from finrl.meta.preprocessor.yahoodownloader import YahooDownloader
from finrl.config import INDICATORS
fe = FeatureEngineer(use_technical_indicator=True,
                     tech_indicator_list = INDICATORS,
                     use_vix=True,
                     use_turbulence=True,
                     user_defined_feature = False)

processed = fe.preprocess_data(portfolio_raw_df)

Successfully added technical indicators


[*********************100%***********************]  1 of 1 completed


Shape of DataFrame:  (2217, 8)
Successfully added vix
Successfully added turbulence index


In [75]:
processed.head()

,date,open,high,low,close,volume,tic,day,macd,boll_ub,boll_lb,rsi_30,cci_30,dx_30,close_30_sma,close_60_sma,vix,turbulence
0,2016-01-04,3536.589111,3538.688965,3295.740967,3296.258057,184400,000001.SS,0,0.0,3304.071949,3279.897045,0.0,-66.666667,100.0,3296.258057,3296.258057,20.700001,0.0
1,2016-01-04,7.675000,7.675000,7.675000,7.492187,0,000035.SZ,0,0.0,3304.071949,3279.897045,0.0,-66.666667,100.0,7.492187,7.492187,20.700001,0.0
2,2016-01-04,35.896500,35.896500,32.266762,31.373688,22430301,000938.SZ,0,0.0,3304.071949,3279.897045,0.0,-66.666667,100.0,31.373688,31.373688,20.700001,0.0
3,2016-01-04,19.299999,19.365000,17.504999,17.414419,16787116,002261.SZ,0,0.0,3304.071949,3279.897045,0.0,-66.666667,100.0,17.414419,17.414419,20.700001,0.0
4,2016-01-04,10.673469,10.948979,9.693877,9.767204,46021099,600547.SS,0,0.0,3304.071949,3279.897045,0.0,-66.666667,100.0,9.767204,9.767204,20.700001,0.0


In [76]:
import sys
# 添加环境路径
sys.path.append('/Users/pu17/Documents/stock/stock_price_prediction')
from feature.ch_feature_engineer import ChFeatureEngineer
ch_fe = ChFeatureEngineer()
# 连接到数据库
portfolio_raw_df = ch_fe.preprocess_data(processed)
print(portfolio_raw_df.head())

2024-10-26 13:17:37,775 - root - INFO - Successfully connected to MySQL database
2024-10-26 13:17:38,040 - root - INFO - Successfully connected to MySQL database
/Users/pu17/Documents/stock/stock_price_prediction/feature/mysqlhandler.py:275: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, self.connection)
2024-10-26 13:17:38,195 - root - INFO - Data successfully fetched from table holiday_features_table between 2014-01-02 and 2024-10-25
2024-10-26 13:17:38,313 - root - INFO - Data successfully fetched from table market_breadth_table between 2015-01-05 and 2024-10-24
2024-10-26 13:17:38,901 - root - INFO - Data successfully fetched from table moneyflow_summary_table between 2015-01-05 and 2024-10-24
2024-10-26 13:17:56,634 - root - INFO - Data successfully fetched from table market_breadth_industry_table betwee

         date         open         high          low        close  volume  \
0  2016-01-04  3536.589111  3538.688965  3295.740967  3296.258057  184400   
1  2016-01-05  3196.650879  3328.138916  3189.604980  3287.710938  266900   
2  2016-01-06  3291.195068  3362.974121  3288.933105  3361.840088  238900   
3  2016-01-07  3309.656982  3309.656982  3115.885010  3125.001953   70600   
4  2016-01-08  3194.625000  3235.450928  3056.877930  3186.412109  286400   

         tic  day      macd      boll_ub  ...  net_md_vol  net_md_amount  \
0  000001.SS    0  0.000000  3304.071949  ...  10481264.0     1428415.80   
1  000001.SS    1 -0.191762  3304.071949  ...   4574474.0      540707.86   
2  000001.SS    2  2.054397  3396.383552  ...  -4734752.0     -251477.90   
3  000001.SS    3 -5.525304  3469.167690  ...   2376686.0      278513.30   
4  000001.SS    4 -7.084027  3440.462154  ...   -225222.0      200576.40   

   net_lg_vol  net_lg_amount  net_elg_vol  net_elg_amount  net_sm_pct  \
0 -1823

In [78]:
# print(portfolio_raw_df.isnull().sum())  # 打印每列中缺失值的数量
portfolio_raw_df = portfolio_raw_df.fillna(0)
print(portfolio_raw_df.isnull().sum()) 

date                            0
open                            0
high                            0
low                             0
close                           0
volume                          0
tic                             0
day                             0
macd                            0
boll_ub                         0
boll_lb                         0
rsi_30                          0
cci_30                          0
dx_30                           0
close_30_sma                    0
close_60_sma                    0
vix                             0
turbulence                      0
spring_festival_pre_holiday     0
spring_festival_post_holiday    0
labor_day_pre_holiday           0
labor_day_post_holiday          0
national_day_pre_holiday        0
national_day_post_holiday       0
dayofmonth                      0
dayofyear                       0
up_count                        0
down_count                      0
up_down_ratio                   0
market_breadth

In [80]:
portfolio_raw_df.head()
# # 将日期时间字符串转换为日期对象，只保留日期部分
# portfolio_raw_df['date'] = portfolio_raw_df.to_datetime(portfolio_raw_df['date'], format='%Y-%m-%d %H:%M:%S').dt.date

,date,open,high,low,close,volume,tic,day,macd,boll_ub,...,net_md_vol,net_md_amount,net_lg_vol,net_lg_amount,net_elg_vol,net_elg_amount,net_sm_pct,net_md_pct,net_lg_pct,net_elg_pct
0,2016-01-04,3536.589111,3538.688965,3295.740967,3296.258057,184400,000001.SS,0,0.000000,3304.071949,...,10481264.0,1428415.80,-18232668.0,-2203051.06,-26384964.0,-3346648.20,0.085997,0.029806,-0.045970,-0.069833
1,2016-01-05,3196.650879,3328.138916,3189.604980,3287.710938,266900,000001.SS,1,-0.191762,3304.071949,...,4574474.0,540707.86,-13749686.0,-1657526.56,-13007012.0,-1371061.34,0.038073,0.008275,-0.025366,-0.020982
2,2016-01-06,3291.195068,3362.974121,3288.933105,3361.840088,238900,000001.SS,2,2.054397,3396.383552,...,-4734752.0,-251477.90,-2362906.0,-392511.22,11979202.0,631894.02,0.000213,-0.004428,-0.006911,0.011125
3,2016-01-07,3309.656982,3309.656982,3115.885010,3125.001953,70600,000001.SS,3,-5.525304,3469.167690,...,2376686.0,278513.30,-7192582.0,-832471.58,-5050944.0,-571290.86,0.070690,0.017497,-0.052297,-0.035889
4,2016-01-08,3194.625000,3235.450928,3056.877930,3186.412109,286400,000001.SS,4,-7.084027,3440.462154,...,-225222.0,200576.40,-8702142.0,-1042594.50,2457376.0,-281744.56,0.017438,0.003112,-0.016179,-0.004372


### Normalize Data

We normalize the data dividing the time series of each stock by its maximum value, so that the dataframe contains values between 0 and 1.

In [81]:
portfolio_norm_df = GroupByScaler(by="tic", scaler=MaxAbsScaler).fit_transform(portfolio_raw_df)
portfolio_norm_df

/Users/pu17/miniconda3/envs/finrobot/lib/python3.10/site-packages/finrl/meta/preprocessor/preprocessors.py:101: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[0.14038828 0.20319756 0.18188047 ... 0.50871717 0.43768557 0.49516559]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  X.loc[select_mask, self.columns] = self.scalers[value].transform(
/Users/pu17/miniconda3/envs/finrobot/lib/python3.10/site-packages/finrl/meta/preprocessor/preprocessors.py:101: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[0.   0.25 0.5  ... 0.   0.25 0.5 ]' has dtype incompatible with int32, please explicitly cast to a compatible dtype first.
  X.loc[select_mask, self.columns] = self.scalers[value].transform(
/Users/pu17/miniconda3/envs/finrobot/lib/python3.10/site-packages/finrl/meta/preprocessor/preproce

,date,open,high,low,close,volume,tic,day,macd,boll_ub,...,net_md_vol,net_md_amount,net_lg_vol,net_lg_amount,net_elg_vol,net_elg_amount,net_sm_pct,net_md_pct,net_lg_pct,net_elg_pct
0,2016-01-04,0.950418,0.948281,0.892473,0.887195,0.140388,000001.SS,0.00,0.000000,0.879068,...,0.464375,0.535060,-0.416632,-0.548929,-0.402864,-0.555478,0.684890,0.511648,-0.644327,-0.522741
1,2016-01-05,0.859064,0.891859,0.863731,0.884895,0.203198,000001.SS,0.25,-0.001391,0.879068,...,0.202673,0.202540,-0.314192,-0.413002,-0.198600,-0.227569,0.303220,0.142043,-0.355535,-0.157063
2,2016-01-06,0.884471,0.901194,0.890629,0.904847,0.181880,000001.SS,0.50,0.014898,0.903628,...,-0.209774,-0.094199,-0.053994,-0.097801,0.182907,0.104882,0.001696,-0.076004,-0.096862,0.083280
3,2016-01-07,0.889433,0.886906,0.843768,0.841101,0.053750,000001.SS,0.75,-0.040068,0.922993,...,0.105300,0.104326,-0.164357,-0.207425,-0.077121,-0.094823,0.562983,0.300345,-0.733009,-0.268653
4,2016-01-08,0.858519,0.867021,0.827790,0.857630,0.218043,000001.SS,1.00,-0.051372,0.915356,...,-0.009979,0.075132,-0.198851,-0.259781,0.037521,-0.046764,0.138880,0.053429,-0.226765,-0.032727
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14450,2024-10-17,0.536171,0.514620,0.539747,0.510853,0.153915,603000.SS,0.75,0.196177,0.007601,...,0.045129,0.025908,-0.023263,-0.020477,0.017020,0.010061,-0.091667,0.078213,-0.029575,0.022473
14451,2024-10-18,0.519439,0.524912,0.528608,0.522583,0.239651,603000.SS,1.00,0.188233,0.007626,...,-0.169806,-0.098880,-0.061676,-0.055531,-0.000784,-0.000552,0.213954,-0.191395,-0.051424,-0.000790
14452,2024-10-21,0.541339,0.532164,0.554177,0.534073,0.201463,603000.SS,0.00,0.187515,0.007655,...,-0.052708,-0.031820,-0.011294,-0.010533,0.007601,0.004768,0.051374,-0.071005,-0.011245,0.007872
14453,2024-10-22,0.546506,0.574035,0.561013,0.587457,0.667018,603000.SS,0.25,0.220364,0.007767,...,-0.738340,-0.482502,-0.029424,-0.036295,0.787662,0.512391,-0.323660,-0.303442,-0.010920,0.238420


In [82]:
# 确保日期列是字符串类型

# 将日期时间字符串转换为日期对象，只保留日期部分
# portfolio_norm_df['date'] = portfolio_norm_df.to_datetime(portfolio_norm_df['date'], format='%Y-%m-%d %H:%M:%S').dt.date

# 如果只想保留日期部分作为字符串
portfolio_norm_df['date'] = portfolio_norm_df['date'].astype(str)

df_portfolio=portfolio_norm_df

df_portfolio_train = df_portfolio[(df_portfolio["date"] >= "2018-01-01") & (df_portfolio["date"] < "2024-02-01")]
df_portfolio_2023 = df_portfolio[(df_portfolio["date"] >= "2024-02-01") & (df_portfolio["date"] < "2024-06-31")]
df_portfolio_2024 = df_portfolio[(df_portfolio["date"] >= "2024-07-01") & (df_portfolio["date"] <= "2024-10-23")]

In [83]:
df_portfolio.to_csv('df_portfolio.csv', index=False)

### Instantiate Environment

Using the `PortfolioOptimizationEnv`, it's easy to instantiate a portfolio optimization environment for reinforcement learning agents. In the example below, we use the dataframe created before to start an environment.

In [48]:
environment = PortfolioOptimizationEnv(
        df_portfolio_train,
        initial_amount=100000,
        comission_fee_pct=0.0025,
        time_window=10,
        features=['open', 'high', 'low', 'close', 'volume', 'day', 'macd',
       'boll_ub', 'boll_lb', 'rsi_30', 'cci_30', 'dx_30', 'close_30_sma',
       'close_60_sma', 'vix', 'turbulence', 'up_count', 'down_count',
       'up_down_ratio', 'market_breadth', 'net_mf_vol', 'net_mf_amount',
       'net_sm_vol', 'net_sm_amount', 'net_md_vol', 'net_md_amount',
       'net_lg_vol', 'net_lg_amount', 'net_elg_vol', 'net_elg_amount',
       'net_sm_pct', 'net_md_pct', 'net_lg_pct', 'net_elg_pct',
       'spring_festival_pre_holiday', 'spring_festival_post_holiday',
       'labor_day_pre_holiday', 'labor_day_post_holiday',
       'national_day_pre_holiday', 'national_day_post_holiday', 'dayofmonth',
       'dayofyear'],
        normalize_df=None
    )

### Instantiate Model

Now, we can instantiate the model using FinRL API. In this example, we are going to use the EIIE architecture introduced by Jiang et. al.

:exclamation: **Note:** Remember to set the architecture's `time_window` parameter with the same value of the environment's `time_window`.

In [49]:
# set PolicyGradient parameters
model_kwargs = {
    "lr": 0.001,
    "policy": EIIE,
}

# here, we can set EIIE's parameters
policy_kwargs = {
    "k_size": 3,
    "time_window": 10,
    "initial_features":42,
}

model = DRLAgent(environment).get_model("pg", device, model_kwargs, policy_kwargs)

### Train Model

In [ ]:
DRLAgent.train_model(model, episodes=260)

### Save Model

In [52]:
torch.save(model.train_policy.state_dict(), "policy_EIIE2.pt")

## Test Model

### Instantiate different environments

Since we have three different periods of time, we need three different environments instantiated to simulate them.

In [53]:
environment_2023 = PortfolioOptimizationEnv(
    df_portfolio_2023,
    initial_amount=100000,
    comission_fee_pct=0.0025,
    time_window=10,
    features=['open', 'high', 'low', 'close', 'volume', 'day', 'macd',
       'boll_ub', 'boll_lb', 'rsi_30', 'cci_30', 'dx_30', 'close_30_sma',
       'close_60_sma', 'vix', 'turbulence', 'up_count', 'down_count',
       'up_down_ratio', 'market_breadth', 'net_mf_vol', 'net_mf_amount',
       'net_sm_vol', 'net_sm_amount', 'net_md_vol', 'net_md_amount',
       'net_lg_vol', 'net_lg_amount', 'net_elg_vol', 'net_elg_amount',
       'net_sm_pct', 'net_md_pct', 'net_lg_pct', 'net_elg_pct',
       'spring_festival_pre_holiday', 'spring_festival_post_holiday',
       'labor_day_pre_holiday', 'labor_day_post_holiday',
       'national_day_pre_holiday', 'national_day_post_holiday', 'dayofmonth',
       'dayofyear'],
        #        "macd",
        #    "boll_ub", "boll_lb", "rsi_30", "cci_30", "dx_30", "close_30_sma",
        #    "close_60_sma", "vix", "turbulence", "up_count",
        #    "down_count", "up_down_ratio", "market_breadth", "net_mf_vol",
        #    "net_mf_amount", "net_sm_vol", "net_sm_amount", "net_md_vol",
        #    "net_md_amount", "net_lg_vol", "net_lg_amount", "net_elg_vol",
        #    "net_elg_amount", "net_sm_pct", "net_md_pct", "net_lg_pct",
        #    "net_elg_pct", "spring_festival_pre_holiday",
        #    "spring_festival_post_holiday", "labor_day_pre_holiday",
        #    "labor_day_post_holiday", "national_day_pre_holiday",
        #    "national_day_post_holiday", "dayofmonth", "dayofyear"],
    normalize_df=None
)

environment_2024 = PortfolioOptimizationEnv(
    df_portfolio_2024,
    initial_amount=100000,
    comission_fee_pct=0.0025,
    time_window=10,
    features=['open', 'high', 'low', 'close', 'volume', 'day', 'macd',
       'boll_ub', 'boll_lb', 'rsi_30', 'cci_30', 'dx_30', 'close_30_sma',
       'close_60_sma', 'vix', 'turbulence', 'up_count', 'down_count',
       'up_down_ratio', 'market_breadth', 'net_mf_vol', 'net_mf_amount',
       'net_sm_vol', 'net_sm_amount', 'net_md_vol', 'net_md_amount',
       'net_lg_vol', 'net_lg_amount', 'net_elg_vol', 'net_elg_amount',
       'net_sm_pct', 'net_md_pct', 'net_lg_pct', 'net_elg_pct',
       'spring_festival_pre_holiday', 'spring_festival_post_holiday',
       'labor_day_pre_holiday', 'labor_day_post_holiday',
       'national_day_pre_holiday', 'national_day_post_holiday', 'dayofmonth',
       'dayofyear'],
            #   , "macd",
        #    "boll_ub", "boll_lb", "rsi_30", "cci_30", "dx_30", "close_30_sma",
        #    "close_60_sma", "vix", "turbulence", "up_count",
        #    "down_count", "up_down_ratio", "market_breadth", "net_mf_vol",
        #    "net_mf_amount", "net_sm_vol", "net_sm_amount", "net_md_vol",
        #    "net_md_amount", "net_lg_vol", "net_lg_amount", "net_elg_vol",
        #    "net_elg_amount", "net_sm_pct", "net_md_pct", "net_lg_pct",
        #    "net_elg_pct", "spring_festival_pre_holiday",
        #    "spring_festival_post_holiday", "labor_day_pre_holiday",
        #    "labor_day_post_holiday", "national_day_pre_holiday",
        #    "national_day_post_holiday", "dayofmonth", "dayofyear"],
    normalize_df=None
)

### Test EIIE architecture
Now, we can test the EIIE architecture in the three different test periods. It's important no note that, in this code, we load the saved policy even though it's not necessary just to show how to save and load your model.

In [ ]:
import pandas as pd
EIIE_results = {
    "training": environment._asset_memory["final"],
    "2023": {},
    "2024": {}
}

# instantiate an architecture with the same arguments used in training
# and load with load_state_dict.
policy = EIIE(time_window=10, initial_features=42,device=device)
policy.load_state_dict(torch.load("policy_EIIE2.pt"))


# 2021
DRLAgent.DRL_validation(model, environment_2023, policy=policy)
EIIE_results["2023"]["value"] = environment_2023._asset_memory["final"]

# 2022
DRLAgent.DRL_validation(model, environment_2024, policy=policy)
EIIE_results["2024"]["value"] = environment_2024._asset_memory["final"]
# 打印环境中保存的动作
print("Actions memory:",pd.DataFrame(environment_2023._actions_memory))

In [ ]:
for i, action in enumerate(environment_2024._actions_memory):
    if not np.isnan(action).all():  # 如果 action 中不全是 NaN
        print(f"Action at step {i}: {action}")

### Test Uniform Buy and Hold
For comparison, we will also test the performance of a uniform buy and hold strategy. In this strategy, the portfolio has no remaining cash and the same percentage of money is allocated in each asset.

In [ ]:
UBAH_results = {
    "train": {},
    "2023": {},
    "2024": {}
}

PORTFOLIO_SIZE = 9

# train period
terminated = False
environment.reset()
while not terminated:
    action = [0] + [1/PORTFOLIO_SIZE] * PORTFOLIO_SIZE
    _, _, terminated, _ = environment.step(action)
UBAH_results["train"]["value"] = environment._asset_memory["final"]


# 2021
terminated = False
environment_2023.reset()
while not terminated:
    action = [0] + [1/PORTFOLIO_SIZE] * PORTFOLIO_SIZE
    _, _, terminated, _ = environment_2023.step(action)
UBAH_results["2023"]["value"] = environment_2023._asset_memory["final"]

# 2022
terminated = False
environment_2024.reset()
while not terminated:
    action = [0] + [1/PORTFOLIO_SIZE] * PORTFOLIO_SIZE
    _, _, terminated, _ = environment_2024.step(action)
UBAH_results["2024"]["value"] = environment_2024._asset_memory["final"]

### Plot graphics

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline 

plt.plot(UBAH_results["train"]["value"], label="Buy and Hold")
plt.plot(EIIE_results["training"], label="EIIE")

plt.xlabel("Days")
plt.ylabel("Portfolio Value")
plt.title("Performance in training period")
plt.legend()

plt.show()

In [ ]:
plt.plot(UBAH_results["2023"]["value"], label="Buy and Hold")
plt.plot(EIIE_results["2023"]["value"], label="EIIE")

plt.xlabel("Days")
plt.ylabel("Portfolio Value")
plt.title("Performance in 2020")
plt.legend()

plt.show()

In [ ]:
plt.plot(UBAH_results["2024"]["value"], label="Buy and Hold")
plt.plot(EIIE_results["2024"]["value"], label="EIIE")

plt.xlabel("Days")
plt.ylabel("Portfolio Value")
plt.title("Performance in 2021")
plt.legend()

plt.show()

In [ ]:
plt.plot(UBAH_results["2022"]["value"], label="Buy and Hold")
plt.plot(EIIE_results["2022"]["value"], label="EIIE")

plt.xlabel("Days")
plt.ylabel("Portfolio Value")
plt.title("Performance in 2022")
plt.legend()

plt.show()

We can see that the agent is able to learn a good policy but its performance is worse the more the test period advances into the future. To get a better performance in 2022, for example, the agent should probably be trained again using more recent data.